In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Needed to import from the enderscope library

import os

os.chdir("..")

In [3]:
import time
import io
from threading import Condition

from rich.pretty import pprint

from picamera2.encoders import JpegEncoder
from picamera2.outputs import FileOutput
from picamera2 import Picamera2, Preview
from libcamera import Transform, controls

import panel as pn

from enderleaf.image import Rectangle

In [4]:
a = Rectangle(0, 100, 0, 100)
pprint(a)
pprint(a.shrink(10, 10))

left:0|right:100|top:0|bottom:100

left:45.0|right:55.0|top:45.0|bottom:55.0

In [5]:
pn.extension()

/tmp/ipykernel_20800/2662542492.py:1: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


In [ ]:
class StreamingOutput(io.BufferedIOBase):
    def __init__(self, panel_output):
        # self.frame = None
        # self.condition = Condition()
        self.panel_output = panel_output

    def write(self, buf):
        # print("tttttt")
        self.panel_output.object = buf
        # with self.condition:
        #     self.frame = buf
        #     self.condition.notify_all()

In [7]:
pane = pn.pane.Placeholder("toto", width=960, height=540)
pane

Placeholder(str, _pane=Markdown, height=540, sizing_mode='fixed', width=960)

In [8]:
# with Picamera2() as picam2:
picam2 = Picamera2()
pprint(picam2.sensor_resolution)
mode = picam2.sensor_modes[2]
pprint(mode)
config = picam2.create_video_configuration(
    main={"size": mode["size"]}, lores={"size": (320, 240)}, encode="lores"
)
picam2.configure(config)
picam2.start_recording(
    JpegEncoder(), FileOutput(StreamingOutput(panel_output=pane))
)
picam2.autofocus_cycle()
time.sleep(4)
new_size = 1000
window = (
    Rectangle(top=0, left=0, bottom=2592, right=4608)
    .shrink(
        new_width=new_size, new_height=new_size * mode["size"][1] / mode["size"][0]
    )
    .ensure_int()
)
pprint(window)
picam2.set_controls(
    {"ScalerCrop": (window.left, window.right, window.width, window.height)}
)
pprint(picam2.controls)
time.sleep(3)
picam2.stop_encoder()
picam2.close()

[7:04:32.584646745] [20800]  INFO Camera camera_manager.cpp:340 libcamera v0.7.0+rpt20260205


(4608, 2592)

[7:04:32.598609575] [20822]  INFO RPI pisp.cpp:720 libpisp version 1.3.0
[7:04:32.614035972] [20822]  INFO IPAProxy ipa_proxy.cpp:180 Using tuning file /usr/share/libcamera/ipa/rpi/pisp/imx708.json
[7:04:32.622681713] [20822]  INFO Camera camera_manager.cpp:223 Adding camera '/base/axi/pcie@1000120000/rp1/i2c@80000/imx708@1a' for pipeline handler rpi/pisp
[7:04:32.622716028] [20822]  INFO RPI pisp.cpp:1181 Registered camera /base/axi/pcie@1000120000/rp1/i2c@80000/imx708@1a to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[7:04:32.630256422] [20800]  INFO Camera camera.cpp:1215 configuring streams: (0) 640x480-XBGR8888/sRGB (1) 1536x864-BGGR_PISP_COMP1/RAW
[7:04:32.630396292] [20822]  INFO RPI pisp.cpp:1485 Sensor: /base/axi/pcie@1000120000/rp1/i2c@80000/imx708@1a - Selected sensor format: 1536x864-SBGGR10_1X10/RAW - Selected CFE format: 1536x864-PC1B/RAW
[7:04:32.677477796] [20800]  INFO Camera camera.cpp:1215 configuring streams: (0) 640x480-XBGR8888/

{
│   'format': SRGGB10_CSI2P,
│   'unpacked': 'SRGGB10',
│   'bit_depth': 10,
│   'size': (4608, 2592),
│   'fps': 14.35,
│   'crop_limits': (0, 0, 4608, 2592),
│   'exposure_limits': (26, 220416802, 20000)
}

[7:04:32.741222453] [20800]  INFO Camera camera.cpp:1215 configuring streams: (0) 4608x2592-XBGR8888/Rec709/Rec709/None/Full (1) 320x240-YUV420/Rec709 (2) 4608x2592-BGGR_PISP_COMP1/RAW
[7:04:32.741447267] [20822]  INFO RPI pisp.cpp:1485 Sensor: /base/axi/pcie@1000120000/rp1/i2c@80000/imx708@1a - Selected sensor format: 4608x2592-SBGGR10_1X10/RAW - Selected CFE format: 4608x2592-PC1B/RAW


left:1804|right:2804|top:1015|bottom:1577

<Controls: {'ScalerCrop': (1804, 2804, 1000, 562)}>

In [9]:
picam2.close()
picam2 = None